In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from nltk.tokenize import word_tokenize
import nltk

# Assicurati di avere i token di NLTK per il tokenization
nltk.download('punkt')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\066321758\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:

# Dataset di esempio
corpus = [
    "Questo è un esempio di generazione di testo",
    "Le reti neurali sono molto potenti per il processing del linguaggio naturale",
    "Le RNN possono essere utilizzate per la previsione della parola successiva",
    "Questo è un esempio semplice di come usare una RNN per generare parole"
]

# Preprocessamento del testo: Tokenizzazione
def preprocess_text(corpus):
    # Tokenizza il testo e unisci tutto in un'unica lista di parole
    text = ' '.join(corpus)
    tokens = word_tokenize(text.lower())  # Tokenizzazione
    return tokens

tokens = preprocess_text(corpus)

# Creazione di un vocabolario tramite Tokenizer

## Creiamo un oggetto Tokenizer di Keras.
# Il Tokenizer serve a costruire un vocabolario numerico del testo:
# a ogni parola viene associato un indice intero.
tokenizer = Tokenizer()

# Analizziamo i token presenti nel corpus e costruiamo il vocabolario.
# [tokens] racchiude la lista dei token all'interno di una lista più esterna,
# perché il metodo fit_on_texts() lavora su uno o più testi.
# Durante questa operazione il Tokenizer:
# - individua le parole presenti;
# - conta la frequenza di ciascuna parola;
# - assegna un indice numerico a ogni parola;
# - salva la corrispondenza parola-indice in tokenizer.word_index.
tokenizer.fit_on_texts([tokens])

# Contiamo quante parole distinte sono presenti nel vocabolario.
# len(tokenizer.word_index) restituisce il numero delle parole conosciute
# dal Tokenizer.
# Aggiungiamo 1 perché gli indici delle parole partono normalmente da 1,
# mentre l'indice 0 viene lasciato libero, ad esempio per il padding.
# total_words rappresenta quindi il numero di classi che il modello dovrà gestire.
total_words = len(tokenizer.word_index) + 1


# Visualizziamo il vocabolario creato dal Tokenizer.
# Il dizionario associa ogni parola al suo indice numerico.
print("Vocabolario:", tokenizer.word_index)

# Visualizziamo il numero totale di parole distinte,
# considerando anche l'indice 0 riservato.
print("Numero totale di parole:", total_words)


Vocabolario: {'di': 1, 'per': 2, 'questo': 3, 'è': 4, 'un': 5, 'esempio': 6, 'le': 7, 'rnn': 8, 'generazione': 9, 'testo': 10, 'reti': 11, 'neurali': 12, 'sono': 13, 'molto': 14, 'potenti': 15, 'il': 16, 'processing': 17, 'del': 18, 'linguaggio': 19, 'naturale': 20, 'possono': 21, 'essere': 22, 'utilizzate': 23, 'la': 24, 'previsione': 25, 'della': 26, 'parola': 27, 'successiva': 28, 'semplice': 29, 'come': 30, 'usare': 31, 'una': 32, 'generare': 33, 'parole': 34}
Numero totale di parole: 35


In [3]:

# Creazione delle sequenze di input e target
input_sequences = []
for i in range(1, len(tokens)):
    n_gram_sequence = tokens[i-1:i+1]  # Crea una sequenza di due parole
    # Esempio:
    # ['questo', 'è']
    input_sequences.append(n_gram_sequence)

# Creazione degli input e degli output per l'addestramento
X = []
y = []

for seq in input_sequences:
    #Convertiamo nell'indice numerico
    prev_word = tokenizer.texts_to_sequences([seq[0]])  # Parola precedente
    next_word = tokenizer.texts_to_sequences([seq[1]])  # Parola successiva
    
    # Verifica che entrambe le parole siano presenti nel vocabolario
    if prev_word and next_word and len(prev_word[0]) > 0 and len(next_word[0]) > 0:
        X.append(prev_word[0][0])  # Aggiungiamo alla lista degli input X
        y.append(next_word[0][0])  #  Aggiungiamo alla lista dei target y.

# Conversione degli input e degli output in array NumPy
X = np.array(X)
y = np.array(y)

# Reshape di X per aggiungere la dimensione "timesteps"
X = X.reshape((X.shape[0], 1))  # Ogni sequenza ha un solo passo temporale (parola)

# Convertiamo y in una rappresentazione one-hot encoded
y = to_categorical(y, num_classes=total_words)

In [4]:
# ============================================================
# VISUALIZZAZIONE DI X DOPO IL RESHAPE
# ============================================================

# Mostriamo i primi 10 esempi di X dopo il reshape.
# Ogni riga contiene l'indice numerico di una parola.
print("Primi 10 elementi di X dopo il reshape:")
print(X[:10])

# Mostriamo la forma di X.
# Il primo valore indica il numero di esempi.
# Il secondo valore indica che ogni sequenza contiene
# un solo timestep, cioè una sola parola.
print("\nForma di X:", X.shape)


# ============================================================
# VISUALIZZAZIONE DI y PRIMA DELLA ONE-HOT ENCODING
# ============================================================

# In questa fase y contiene ancora gli indici numerici
# delle parole successive che la rete dovrà prevedere.
print("\nPrimi 10 elementi di y prima della one-hot encoding:")
print(y[:10])

# Mostriamo la forma di y prima della conversione.
print("\nForma di y prima della one-hot encoding:", y.shape)

Primi 10 elementi di X dopo il reshape:
[[ 3]
 [ 4]
 [ 5]
 [ 6]
 [ 1]
 [ 9]
 [ 1]
 [10]
 [ 7]
 [11]]

Forma di X: (43, 1)

Primi 10 elementi di y prima della one-hot encoding:
[[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0

In [6]:

# Costruzione del modello RNN classico
model = Sequential()
model.add(Embedding(total_words, 50, input_length=1))  # L'input è una singola parola
model.add(SimpleRNN(100, return_sequences=False))  # RNN classica
model.add(Dense(total_words, activation='softmax'))

# Compilazione del modello
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Addestramento del modello
model.fit(X, y, epochs=100, verbose=1)


# Questo blocco costruisce una RNN classica per prevedere la parola successiva.
# L'Embedding trasforma l'indice della parola in un vettore di 50 numeri.
# La SimpleRNN elabora questo vettore utilizzando 100 neuroni ricorrenti.
# Il livello Dense, con attivazione softmax, restituisce una probabilità
# per ogni parola del vocabolario.
# Successivamente, compile() prepara il modello all'addestramento,
# mentre fit() addestra la rete per 100 epoche usando X come input
# e y come parola successiva corretta.


Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.0000e+00 - loss: 3.5601
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.0000e+00 - loss: 3.5429
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.0930 - loss: 3.5286
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.2093 - loss: 3.5146
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4186 - loss: 3.5007
Epoch 6/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5814 - loss: 3.4869
Epoch 7/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.6279 - loss: 3.4727
Epoch 8/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.6512 - loss: 3.4587
Epoch 9/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.6744 - loss: 3.4440
Epoch 10/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.6744 - loss: 3.4290
Epoch 11/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.6744 - loss: 3.4131
Epoch 12/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6

In [ ]:
import numpy as np

# model: il modello RNN già addestrato.
# tokenizer: lo strumento che converte le parole in indici numerici
#            e gli indici numerici nelle parole corrispondenti.
# text: il testo utilizzato come input per la previsione.
def predict_next_word(model, tokenizer, text):

    # Convertiamo il testo in minuscolo e lo dividiamo in parole
    # utilizzando split().
    #
    # Successivamente, il tokenizer converte ogni parola
    # nel relativo indice numerico presente nel vocabolario.
    #
    # Esempio:
    # "questo è un esempio"
    # diventa:
    # [[4, 5, 6, 7]]
    tokenized_input = tokenizer.texts_to_sequences(
        [text.lower().split()]
    )

    # Convertiamo la sequenza numerica in un array NumPy,
    # così da poterla utilizzare come input del modello.
    tokenized_input = np.array(tokenized_input)

    # Adattiamo la forma dell'array a quella richiesta dalla RNN.
    #
    # Il primo valore, 1, indica che stiamo passando
    # una sola sequenza di testo.
    #
    # tokenized_input.shape[1] indica il numero di parole
    # presenti nella sequenza di input.
    #
    # Esempio:
    # [[4, 5, 6, 7]]
    # ha forma (1, 4).
    tokenized_input = tokenized_input.reshape(
        (1, tokenized_input.shape[1])
    )

    # Passiamo l'input numerico al modello RNN già addestrato.
    #
    # Il modello restituisce una probabilità per ogni parola
    # presente nel vocabolario.
    predicted_probs = model.predict(tokenized_input)

    # Individuiamo l'indice associato alla probabilità più alta.
    #
    # np.argmax() restituisce la posizione del valore massimo
    # all'interno del vettore delle probabilità.
    predicted_word_index = np.argmax(predicted_probs)

    # Convertiamo l'indice numerico previsto nella parola
    # corrispondente utilizzando il dizionario inverso del tokenizer.
    #
    # Esempio:
    # 5 → "è"
    predicted_word = tokenizer.index_word[predicted_word_index]

    # Restituiamo la parola successiva prevista dal modello.
    return predicted_word

In [12]:
input_text = "le reti"
predicted_word = predict_next_word(model, tokenizer, input_text)
print(f"Parola successiva prevista per '{input_text}': {predicted_word}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Parola successiva prevista per 'le reti': neurali
